In [49]:
USE [sample];
GO


Commands completed successfully.

In [50]:
/*
    Best Practice: Stop "rows affected" noise
*/
SET NOCOUNT ON;

SELECT DB_NAME() AS db_name;
DECLARE @Msg NVARCHAR(MAX) = 'Initial Database Context: ' + DB_NAME();
EXEC dbo.spInfo @Msg;
GO


2026-03-26 06:53:05 | INFO | Initial Database Context: sample

db_name
-------
sample 
(1 row)

Timestamp               | Level | Message                         
------------------------+-------+---------------------------------
2026-03-26 06:53:05.877 | INFO  | Initial Database Context: sample
(1 row)

In [51]:
/*
************************************************************************************
    name of the foreign key to use with OBJEC_ID()
    cannot use GO while @FK_NAME is defined
    after the GO the scope dies and with it the variable
************************************************************************************
*/
DECLARE @FK_NAME VARCHAR(128) = 'dbo.FK_tblPerson_tblGender';
DECLARE @Msg NVARCHAR(MAX);

PRINT @FK_NAME

-- check existence of the FK
;WITH fkid
AS
(
    SELECT COALESCE(
                /* 
                    object_id return an integer, 
                    so coalesce must use an integer too !!!
                */
                OBJECT_ID(@FK_NAME, 'F'),               -- if the foreign key does not exist returns null
                0                                       -- if object_id returns null, returns 0
            ) AS FK_ID
)

/*
SELECT 
    FK_ID,
    CASE 
        WHEN FK_ID = 0 THEN 'NOT_FOUND ' + @FK_NAME     -- no || concat operator
        ELSE 'FOUND ' + @FK_NAME                        -- then and else must be of the same type
    END AS 'FOUND FK_ID?'                               -- no dynamic strings for aliases
FROM fkid
*/
SELECT 
    @Msg = CASE 
        WHEN FK_ID = 0 THEN 'NOT_FOUND ' + @FK_NAME     -- no || concat operator
        ELSE 'FOUND ' + @FK_NAME                        -- then and else must be of the same type
    END    
FROM fkid
;
-- GO
EXEC dbo.spInfo @Msg;


dbo.FK_tblPerson_tblGender
2026-03-26 06:53:05 | INFO | NOT_FOUND dbo.FK_tblPerson_tblGender

Timestamp               | Level | Message                             
------------------------+-------+-------------------------------------
2026-03-26 06:53:05.890 | INFO  | NOT_FOUND dbo.FK_tblPerson_tblGender
(1 row)

In [52]:

/*
************************************************************************************
    oneliner
************************************************************************************
*/
DECLARE @FK_NAME VARCHAR(128) = 'dbo.FK_tblPerson_tblGender';
PRINT @FK_NAME
SELECT OBJECT_ID(@FK_NAME, 'F') AS 'oneliner';


dbo.FK_tblPerson_tblGender

oneliner
--------
NULL    
(1 row)

In [53]:
DECLARE @FK_NAME VARCHAR(128) = 'dbo.FK_tblPerson_tblGender';
/*
************************************************************************************
    composite statement
************************************************************************************
*/
DECLARE @FK_ID INT = OBJECT_ID(@FK_NAME, 'F');
SELECT @FK_ID as FK_ID;

SELECT COALESCE(
        CAST(
            OBJECT_ID(@FK_NAME, 'F') AS VARCHAR(20)
        ),
        /*
            or use convert, only mssql
            CONVERT(VARCHAR(10), OBJECT_ID(@FK_NAME, 'F'))
        */
        'not found ' + @FK_NAME
    ) AS 'composite statement'
;


Commands completed successfully.

FK_ID
-----
NULL 
(1 row)

composite statement                 
------------------------------------
not found dbo.FK_tblPerson_tblGender
(1 row)

In [54]:
DECLARE @FK_NAME VARCHAR(128) = 'dbo.FK_tblPerson_tblGender';
/*
************************************************************************************
    create FK if it doesnt exist
************************************************************************************
*/
IF OBJECT_ID(@FK_NAME, 'F') IS NOT NULL
BEGIN
    SELECT 'Foreign Key ' + @FK_NAME + ' already exists.' AS MESSAGE
    UNION ALL
    SELECT 'DROPPING Foreign Key ' + @FK_NAME AS MESSAGE;
    ALTER TABLE [dbo].[tblPerson]
    DROP CONSTRAINT [FK_tblPerson_tblGender];
    SELECT 'Foreign Key ' + @FK_NAME + ' dropped.' AS MESSAGE;
END
SELECT 'CREATING Foreign Key ' + @FK_NAME AS MESSAGE;
ALTER TABLE [dbo].[tblPerson]               -- object do modify
ADD CONSTRAINT [FK_tblPerson_tblGender]     -- name of constraint
FOREIGN KEY ([GenderId])                    -- foreign key column
REFERENCES [dbo].[tblGender] ([ID]);        -- external referenced column

DECLARE @Msg NVARCHAR(MAX) = CONCAT('INFO | Foreign Key ', @FK_NAME, ' created.')
EXEC dbo.spInfo @Msg;


2026-03-26 06:53:05 | INFO | INFO | Foreign Key dbo.FK_tblPerson_tblGender created.

MESSAGE                                        
-----------------------------------------------
CREATING Foreign Key dbo.FK_tblPerson_tblGender
(1 row)

Timestamp               | Level | Message                                               
------------------------+-------+-------------------------------------------------------
2026-03-26 06:53:05.917 | INFO  | INFO | Foreign Key dbo.FK_tblPerson_tblGender created.
(1 row)

In [55]:
/* 
************************************************************************************
    test fk
    cannot add a missing reference to FK
************************************************************************************
*/
BEGIN TRY
    insert into [dbo].[tblPerson]
    VALUES (1, 'name', 'email', 1);
END TRY
BEGIN CATCH
    EXEC dbo.spError 'An error occurred. Execution jumped to the CATCH block.'
    -- UNION ALL
    -- SELECT CAST(ERROR_NUMBER() AS NVARCHAR) + ' - ' + ERROR_MESSAGE() AS MESSAGE
END CATCH
GO


2026-03-26 06:53:52 | ERROR | An error occurred. Execution jumped to the CATCH block.: 547 - The INSERT statement conflicted with the FOREIGN KEY constraint "FK_tblPerson_tblGender". The conflict occurred in database "sample", table "dbo.tblGender", column 'ID'.

Timestamp               | Level | Message                                                                                                                                                                                                 
------------------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2026-03-26 06:53:52.310 | ERROR | An error occurred. Execution jumped to the CATCH block.: 547 - The INSERT statement conflicted with the FOREIGN KEY constraint "FK_tblPerson_tblGender". The conflict occurred in database "sample", table "dbo.tblGender", column 'ID'.
(1 row)